# Load Library

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
from scipy.stats import friedmanchisquare

# Load results

In [1]:
imputation_results_raw_datav2 ='/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/imputation_results_raw_datav2.csv'
imputation_results_seed42_datav2 ='/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/imputation_results_seed42_datav2.csv'
imputation_results_summary_datav2 ='/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/imputation_results_summary_datav2.csv'

In [2]:
downstream_results_raw_datav2 ='/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_results_raw_datav2.csv'
downstream_results_summary_datav2 ='/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_results_summary_datav2.csv'
downstream_auc_diff_vs_full_datav2 ='/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_auc_diff_vs_full_datav2.csv'
downstream_friedman_by_scenario_datav2 ='/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_friedman_by_scenario_datav2.csv'
downstream_pooled_posthoc_datav2 ='/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_pooled_posthoc_datav2.csv'

# 1. Statistical test

In [21]:
from scipy.stats import friedmanchisquare

def friedman_test(df, mechanism, missing_rate, metric="roc_auc"):
    methods = ["Mean", "Median", "KNN", "MICE"]
    samples = [
        df[(df["mechanism"]==mechanism) & (df["missing_rate"]==missing_rate) & (df["method"]==m)]
        .sort_values("seed")[metric].to_numpy()
        for m in methods
    ]
    stat, p = friedmanchisquare(*samples)
    print(f"{mechanism}/{missing_rate:.0%}: Friedman p = {p:.4f}")
    return p

In [22]:
def friedman_test(df, mechanism, missing_rate, metric="roc_auc"):
    """
    Friedman test: compares all 4 imputation methods at once
    (paired by seed, non-parametric, suitable for small n).
    """
    methods = ["Mean", "Median", "KNN", "MICE"]
    samples = [
        df[(df["mechanism"] == mechanism) &
           (df["missing_rate"] == missing_rate) &
           (df["method"] == m)]
        .sort_values("seed")[metric]
        .to_numpy()
        for m in methods
    ]

    stat, p = friedmanchisquare(*samples)
    return {
        "mechanism": mechanism,
        "missing_rate": missing_rate,
        "friedman_stat": stat,
        "p_value": p
    }




In [23]:

friedman_results = []
for mechanism in ["MCAR", "MAR", "MNAR"]:
    for rate in [0.1, 0.3, 0.5]:
        friedman_results.append(friedman_test(downstream_results_df, mechanism, rate))

friedman_df = pd.DataFrame(friedman_results)
friedman_df.to_csv(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/friedman_test_results.csv",
    index=False
)

print("Number of mechanism/rate combinations tested:", len(friedman_df))
print("Number significant (p < 0.05):", (friedman_df["p_value"] < 0.05).sum())
friedman_df.sort_values("p_value")

Number of mechanism/rate combinations tested: 9
Number significant (p < 0.05): 1


,mechanism,missing_rate,friedman_stat,p_value
1,MCAR,0.3,9.24,0.026264
0,MCAR,0.1,6.12,0.105916
2,MCAR,0.5,5.40,0.144744
6,MNAR,0.1,5.16,0.160449
8,MNAR,0.5,4.92,0.177750
4,MAR,0.3,4.20,0.240662
3,MAR,0.1,3.48,0.323365
7,MNAR,0.3,1.32,0.724389
5,MAR,0.5,0.60,0.896432


In [ ]:
def holm_correction(p_values, alpha=0.05):
    p_values = np.array(p_values)
    n = len(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]

    reject_sorted = np.zeros(n, dtype=bool)
    p_corrected_sorted = np.zeros(n)

    for i in range(n):
        p_corrected_sorted[i] = min(sorted_p[i] * (n - i), 1.0)
        # 保证校正后的p值单调不减
        if i > 0:
            p_corrected_sorted[i] = max(p_corrected_sorted[i], p_corrected_sorted[i-1])

    # 一旦某个不显著，后面（更大p值的）全部不显著
    for i in range(n):
        if p_corrected_sorted[i] <= alpha:
            reject_sorted[i] = True
        else:
            break

    reject = np.zeros(n, dtype=bool)
    p_corrected = np.zeros(n)
    reject[order] = reject_sorted
    p_corrected[order] = p_corrected_sorted

    return reject, p_corrected

reject, p_holm = holm_correction(friedman_df["p_value"].values)
friedman_df["p_holm"] = p_holm
friedman_df["significant_holm"] = reject


In [28]:
friedman_df

,mechanism,missing_rate,friedman_stat,p_value,p_holm,significant_holm
0,MCAR,0.1,6.12,0.105916,0.847326,False
1,MCAR,0.3,9.24,0.026264,0.236380,False
2,MCAR,0.5,5.40,0.144744,1.000000,False
3,MAR,0.1,3.48,0.323365,1.000000,False
4,MAR,0.3,4.20,0.240662,1.000000,False
5,MAR,0.5,0.60,0.896432,1.000000,False
6,MNAR,0.1,5.16,0.160449,1.000000,False
7,MNAR,0.3,1.32,0.724389,1.000000,False
8,MNAR,0.5,4.92,0.177750,1.000000,False


In [29]:
friedman_df = pd.DataFrame(friedman_results)
friedman_df.to_csv(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/friedman_test_results.csv",
    index=False
)

# 2. Imputation results vs Downstream performance

In [ ]:
summary_df = (
    downstream_results_df
    .groupby(["mechanism", "missing_rate", "method"])
    .agg(
        accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"), precision_std=("precision", "std"),
        recall_mean=("recall", "mean"), recall_std=("recall", "std"),
        f1_mean=("f1", "mean"), f1_std=("f1", "std"),
        roc_auc_mean=("roc_auc", "mean"), roc_auc_std=("roc_auc", "std"),
    )
    .reset_index()
)

summary_df

,mechanism,missing_rate,method,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Baseline,0.0,Drop_column,0.650459,0.009861,0.108201,0.003144,0.457363,0.012390,0.174980,0.004559,0.584970,0.005395
1,Baseline,0.0,Full_data,0.639225,0.006161,0.108520,0.002825,0.478585,0.017084,0.176913,0.004743,0.592370,0.006143
2,MAR,0.1,KNN,0.639840,0.007178,0.108535,0.003022,0.477685,0.020736,0.176865,0.005226,0.590689,0.005369
3,MAR,0.1,MICE,0.639569,0.005124,0.108652,0.002364,0.478714,0.013631,0.177100,0.003912,0.590884,0.003882
4,MAR,0.1,Mean,0.636150,0.006326,0.108337,0.001591,0.482830,0.015153,0.176955,0.002921,0.592478,0.005586
5,MAR,0.1,Median,0.634754,0.008143,0.107950,0.003135,0.482830,0.015153,0.176437,0.004895,0.591585,0.005347
6,MAR,0.3,KNN,0.640694,0.008111,0.107712,0.002998,0.471511,0.018389,0.175347,0.004939,0.589425,0.004645
7,MAR,0.3,MICE,0.640652,0.005151,0.108051,0.002957,0.473441,0.013284,0.175941,0.004722,0.589534,0.004709
8,MAR,0.3,Mean,0.641861,0.008762,0.108752,0.002640,0.475113,0.008892,0.176976,0.003707,0.591769,0.004854
9,MAR,0.3,Median,0.641986,0.006358,0.108126,0.002667,0.471768,0.020125,0.175916,0.004772,0.590007,0.005521


In [ ]:
comparison_df = imputation_results_df.merge(
    summary_df[["mechanism", "missing_rate", "method", "roc_auc_mean", "roc_auc_std"]],
    on=["mechanism", "missing_rate", "method"]
)

comparison_df = comparison_df.sort_values(["mechanism", "missing_rate", "MAE"])
comparison_df

,mechanism,missing_rate,method,MAE,RMSE,roc_auc_mean,roc_auc_std
0,MAR,0.1,KNN,7.127056,15.865384,0.590689,0.005369
9,MAR,0.1,MICE,7.959261,16.142786,0.590884,0.003882
27,MAR,0.1,Median,13.564738,29.464809,0.591585,0.005347
18,MAR,0.1,Mean,13.663391,27.970665,0.592478,0.005586
1,MAR,0.3,KNN,6.938995,17.389804,0.589425,0.004645
10,MAR,0.3,MICE,7.673284,17.429936,0.589534,0.004709
19,MAR,0.3,Mean,12.836971,27.846714,0.591769,0.004854
28,MAR,0.3,Median,12.949991,29.140020,0.590007,0.005521
2,MAR,0.5,KNN,6.767097,16.341668,0.588188,0.005816
11,MAR,0.5,MICE,7.224448,16.225258,0.588587,0.003706


In [32]:
assert len(comparison_df) == len (imputation_results_df)
print('OK:',len(comparison_df),"row")

OK: 36 row


In [ ]:
from scipy.stats import pearsonr, spearmanr

pearson_r, pearson_p = pearsonr(comparison_df["MAE"], comparison_df["roc_auc_mean"])
spearman_r, spearman_p = spearmanr(comparison_df["MAE"], comparison_df["roc_auc_mean"])

print(f"Pearson r = {pearson_r:.3f}, p = {pearson_p:.3f}")
print(f"Spearman r = {spearman_r:.3f}, p = {spearman_p:.3f}")


Pearson r = 0.207, p = 0.226
Spearman r = 0.264, p = 0.120
